In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, ShortType, ArrayType

In [0]:

cols_to_keep = ['id', 'name', 'popularity', 'duration_ms', 'artists', 'id_artists', 'release_date']

df = (
      spark.table("spotify_project.bronze.spotify_tracks")
      .select(*cols_to_keep)
      .dropDuplicates(["id"])
      .dropna(subset=["id", "name"])
      .filter(F.col("popularity").rlike("^[0-9]+$"))
      .withColumn("popularity", F.col("popularity").cast("int"))
      .filter(F.col("duration_ms").rlike("^[0-9]+$"))
      .withColumn("duration_ms", F.col("duration_ms").cast("long"))
      .filter(F.col("duration_ms") > 0)
      .withColumn(
          "artists",
          F.from_json(F.col("artists"), ArrayType(StringType()))
      )
      .withColumn(
          "artists",
          F.expr("""filter(artists, a -> not rlike(a, '^[0-9]+$') and a != '')""")
      )
      .withColumn("artists", F.array_distinct("artists"))
      .filter(F.size("artists") > 0)
      .withColumn(
          "id_artists",
          F.from_json(F.col("id_artists"), ArrayType(StringType()))
      )
      .withColumn(
          "id_artists",
          F.expr("""filter(id_artists, x -> rlike(x, '^[A-Za-z0-9]+$'))""")
      )
      .filter(F.size("id_artists") > 0)
      .filter(F.col("release_date").rlike("^[0-9]{4}$|^[0-9]{4}-[0-9]{2}-[0-9]{2}$|^[0-9]{4}-[0-9]{2}$"))
      .withColumn(
          "release_date",
          F.when(F.col("release_date").rlike("^[0-9]{4}$"), F.concat(F.col("release_date"), F.lit("-01-01")))
           .when(F.col("release_date").rlike("^[0-9]{4}-[0-9]{2}$"), F.concat(F.col("release_date"), F.lit("-01")))
           .otherwise(F.col("release_date"))
      )
      .filter(F.size("artists") == F.size("id_artists"))
      .select(
          "id", "name", "popularity", "duration_ms", "release_date",
          F.explode(F.arrays_zip("artists", "id_artists")).alias("artist_info")
      )
      .select(
          "id", "name", "popularity", "duration_ms", "release_date",
          F.col("artist_info.artists").alias("artist"),
          F.col("artist_info.id_artists").alias("id_artist")
      )
)

df.write.mode("overwrite").option('overwriteSchema', "true").saveAsTable("spotify_project.silver.track_list")


In [0]:
%sql
SELECT 
    artist,
    id_artist,
    track.name AS track_name,
    track.id AS track_id,
    followers AS follower_count,
    artist_info.popularity AS artist_popularity
FROM spotify_project.silver.track_list track JOIN spotify_project.silver.artists_info artist_info ON artist_info.id = track.id_artist 
WHERE artist_info.name = 'Perfume'